# Building a Python Fixing AI Agent using Lang Frameworks — Google Colab

The same bug-fixing agent, three times. You **read** the first one, and you **write** the parts of
the second and third that no framework can write for you: where a run is allowed to end, and what
to do about a model that has stopped making progress.

| Lesson | Repository | Framework | Reasoning | You write |
|---|---|---|---|---|
| 2 — Agent with no framework | `agentfix-workshop` | none — a `for` loop | no | — (read it, run it) |
| 3 — What about frameworks? | `agentfix-langchain` | LangGraph | no | the graph's routing, and the loop guard |
| 4 — What about thinking? | `agentfix-react` | LangGraph | **yes** | what counts as acting, the idle counter, the nudge choice, the routing tail |

## Important repository rules

Each lesson gets its own clone under `/content`, always from the GitHub **`main` branch**, which
is the exercise branch:

```
/content/agentfix-workshop     read-only tour, no exercises
/content/agentfix-langchain    Stage 1 and Stage 2
/content/agentfix-react        Stage 1
```

This notebook **never checks out a solution branch or tag in your working tree.** Solutions are
only ever read with `git diff` / `git show`, so peeking cannot overwrite your work. Pushing is
disabled on every clone.

The three ideas the whole course is about: **tools**, a **loop** that feeds tool results back to
the model, and a way to know when it is **done** that does not depend on the model's opinion.

## 0. Before you run anything

In Colab choose **Runtime → Change runtime type → GPU**.

You will edit source files directly in Colab:

1. Click the **folder icon** in the left sidebar.
2. Open `content → agentfix-langchain` (or `agentfix-react` for lesson 4).
3. Double-click a `.py` or `.md` file to open it in Colab's editor.
4. Edit and save with **Ctrl+S / Cmd+S**.
5. Return to this notebook and rerun the relevant test cell.

Do **not** use `%%writefile` for the exercises. You are editing the real cloned repository, and
every install below is editable, so a saved file takes effect immediately.

Run the cells in order. Lessons 3 and 4 depend on the setup in sections 1–4 only.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true
!free -g | head -2

## 1. Configuration

Two models, because lesson 4 needs one that **reasons**:

| Lessons | Model | Why |
|---|---|---|
| 2 and 3 | `qwen2.5-coder:1.5b`, derived as `agentfix-qwen` | the small stand-in for Mellum2 **Instruct** |
| 4 | `qwen3:1.7b` | the smallest model that both **thinks** and calls tools |

`qwen2.5-coder:1.5b` has no thinking mode at all. Point lesson 4 at it and every run still
completes — silently, as the Act-only agent from lesson 3 — and "this agent does not reason"
becomes a fact about your setup rather than about the model. `agentgraph doctor` fails rather than
letting that pass quietly.

`MELLUM_MODEL` is the environment variable every edition reads for the Ollama model name.

In [ ]:
%env OLLAMA_CONTEXT_LENGTH=16384

!echo "workshop   https://github.com/jelenadjuric01/agentfix-workshop.git   -> /content/agentfix-workshop"
!echo "langchain  https://github.com/jelenadjuric01/agentfix-langchain.git  -> /content/agentfix-langchain"
!echo "react      https://github.com/jelenadjuric01/agentfix-react.git      -> /content/agentfix-react"
!echo
!echo "branch          main  (the exercise branch, in every repo)"
!echo "lessons 2-3     qwen2.5-coder:1.5b -> agentfix-qwen"
!echo "lesson 4        qwen3:1.7b"
!echo "context length  16384"

## 2. Install and start Ollama

Colab may not include `zstd`, which the current Ollama Linux installer needs. The server is
started in the background and stays available to every later cell.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd pciutils
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; fi
!ollama --version

In [ ]:
!if ! curl -fsS http://127.0.0.1:11434/api/version >/dev/null 2>&1; then nohup ollama serve > /tmp/ollama-colab.log 2>&1 & sleep 4; fi
!curl -fsS http://127.0.0.1:11434/api/version

## 3. Pull both models

The `ollama create` step for lessons 2 and 3 bakes `num_ctx 16384` into a derived model with a
stable name. Too small a context window does not error — it silently truncates the middle of the
agent's history, which looks like a stupid model rather than a misconfiguration.

`qwen3:1.7b` needs no `create` step: the ReAct edition talks to Ollama's native API, which honours
a per-request `num_ctx`.

This cell downloads about 2.5 GB in total, so it is the slow one.

In [ ]:
!ollama pull qwen2.5-coder:1.5b
!printf "FROM qwen2.5-coder:1.5b\nPARAMETER num_ctx 16384\n" > /tmp/Modelfile.agentfix-qwen
!ollama create agentfix-qwen -f /tmp/Modelfile.agentfix-qwen

!ollama pull qwen3:1.7b

!ollama list

In [ ]:
# Smoke test both channels: the Instruct model answers, the Thinking model thinks first.
!ollama run agentfix-qwen "Reply with exactly: READY"
!ollama run qwen3:1.7b "Reply with exactly: READY"

## 4. Clone the three repositories — clean `main` only

**This is intentionally a clean reset.** The cell deletes any previous clone and clones `main`
again, so a stale checkout or an old Colab editor session cannot contaminate your starting point.
It also disables the push URL on each remote, so an accidental `git push` fails while `git fetch`
keeps working.

**Do not rerun this cell after you begin editing** unless you intentionally want to discard your
exercise work.

In [ ]:
%cd /content

!for r in agentfix-workshop agentfix-langchain agentfix-react; do \
    rm -rf /content/$r; \
    git clone -q --branch main https://github.com/jelenadjuric01/$r.git /content/$r; \
    git -C /content/$r fetch -q --all --tags --prune; \
    git -C /content/$r reset -q --hard origin/main; \
    git -C /content/$r clean -qfd; \
    git -C /content/$r remote set-url --push origin DISABLED; \
    echo "$r  branch=$(git -C /content/$r branch --show-current)  HEAD=$(git -C /content/$r rev-parse --short HEAD)  push=$(git -C /content/$r remote get-url --push origin)"; \
  done

### Verify you are on the exercise version

A guardrail. Each exercise file must be byte-for-byte identical to `origin/main`, and the
`EXERCISE(stage-N)` markers must still be there. If these pass, you are definitely starting from
the stubbed version rather than a solution.

Expected: **2** markers in the LangGraph edition (one per stage), **4** in the ReAct edition (all
one stage — one decision split four ways).

In [ ]:
!cd /content/agentfix-langchain && git diff --exit-code origin/main -- src/agentfix/agent/graph.py && echo "OK: langchain graph.py matches origin/main exactly."
!cd /content/agentfix-react     && git diff --exit-code origin/main -- src/agentgraph/agent/graph.py && echo "OK: react graph.py matches origin/main exactly."

!echo
!cd /content/agentfix-langchain && test "$(grep -c 'EXERCISE(stage-' src/agentfix/agent/graph.py)" -eq 2 && echo "OK: langchain has 2 exercise markers."
!cd /content/agentfix-react     && test "$(grep -c 'EXERCISE(stage-' src/agentgraph/agent/graph.py)" -eq 4 && echo "OK: react has 4 exercise markers."

!echo
!cd /content/agentfix-langchain && grep -n "EXERCISE(stage-" src/agentfix/agent/graph.py
!echo
!cd /content/agentfix-react     && grep -n "EXERCISE(stage-" src/agentgraph/agent/graph.py

---

# Lesson 2 — Agent with no framework

Nothing to write here. Read the agent, run it, and form an opinion about which parts of it are
*this project* and which parts are plumbing a framework could own — because lesson 3 answers that
question with code.

An agent is a while-loop around a chat model that can call functions and sees the result. Nothing
more magical than that. The loop in `src/agentfix/agent/loop.py` is about 15 lines; the rest of
`run_agent` is tracing and token accounting.

> **Note on the install.** This edition and the LangGraph edition are both packaged as `agentfix`,
> so installing one replaces the other. That is fine as long as you go through the notebook in
> order — each lesson reinstalls its own. Come back here and rerun this cell if you want the
> no-framework agent again.

In [ ]:
%cd /content/agentfix-workshop
%env MELLUM_MODEL=agentfix-qwen

!python -m pip uninstall -q -y agentfix
!python -m pip install -q -e .
!agentfix doctor

`doctor` should report `[PASS]` on everything except `ram` — Colab has less than 16 GB, which is
exactly why this notebook uses the small models. A `ram` FAIL here is expected and harmless.

### Read the three ideas

`tools/` gives the model actions, `agent/loop.py` decides what happens next, `sandbox/` executes
safely. The stop condition is the one worth reading twice.

In [ ]:
!echo "===== the loop ====="; sed -n '/^def run_agent/,/^    return AgentResult/p' src/agentfix/agent/loop.py | head -60
!echo; echo "===== the stop condition ====="; sed -n '/^def is_done/,/^def /p' src/agentfix/agent/loop.py | head -25

### Run it for real

`--verbose` prints the trace. You should see the model call `run_tests`, look around, write a file,
and run the tests again — that last call is what ends the run.

A 1.5B model does not fix every task. `NOT SOLVED` after ten steps is not a broken setup; it is a
small model, and watching it fail is informative. The second task is the harder one: the bug is
**not** in the file the failing test points at, which is why `list_files` and `read_file` earn
their place.

In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose

In [ ]:
!agentfix solve tasks/workshop/02-invoice --verbose

---

# Lesson 3 — What about frameworks?

The same agent rebuilt on LangGraph for the graph and LangChain for the model and tool interfaces.
The question the lesson exists to answer: *which parts of my agent does a framework actually write
for me?*

**What it gives you:** `ToolNode` runs the calls (dispatch, ordering, unknown tool names, argument
validation, error recovery). `add_messages` makes the history append-only by construction, which
keeps the prompt prefix byte-stable and the server's KV cache valid. Reducers on `AgentState`
accumulate the counters. Callbacks carry the trace. The checkpointer snapshots state after every
node.

**What it does not:** the stop condition, the loop guard, and the step budget — `recursion_limit`
counts node executions, not model turns. Those are your two stages.

In [ ]:
%cd /content/agentfix-langchain
%env MELLUM_MODEL=agentfix-qwen

!python -m pip uninstall -q -y agentfix
!python -m pip install -q -e .
!agentfix doctor

Read `agent/state.py` before you touch `agent/graph.py`. Making the loop's local variables an
explicit typed state is the biggest change the framework asks for, and it costs one thing: a node
returns a *partial* state that LangGraph merges, so `counter += 1` is not something a node can
do — hence the reducers.

One detail worth the whole lesson: checkpointing is only as good as what you put in the state.
While the test verdict lived on the `run_tests` tool, the graph was resumable and the *agent* was
not.

In [ ]:
!cat exercises/README.md

## Stage 1 — where a run is allowed to end

Open `src/agentfix/agent/graph.py` and find `EXERCISE(stage-1)` in **`route_after_agent`**. It
looks at the model turn that just happened and returns `"tools"`, `"nudge"`, or `END`.

This is the only place a run can end **successfully**, and the framework has no opinion about it
whatsoever. What you have to work with: `message.tool_calls` (what the model asked for),
`is_done(state)` (the verdict), `state["step"]` and `max_steps` (the budget).

Four things to get right, and their **order** is as much of the decision as the answers are:

1. Tool calls never end a run on their own — execute them and loop back so the model sees the
   results. This branch deliberately skips `is_done`: "done" belongs on a turn where the model had
   nothing more it wanted to do.
2. On a prose turn the verdict decides. `is_done` reads `state["tests_passed"]`, which only ever
   becomes true by folding a real test result out of a tool answer — so a model that declares
   victory without running the tests is **not** believed.
3. The step budget outranks the nudge, or a stubborn model never stops.
4. Otherwise, nudge it and go again.

The full guide is `exercises/stage_1/README.md`.

In [ ]:
!cat exercises/stage_1/README.md

In [ ]:
!echo "Stage 1 — where to edit:"
!grep -n -B18 -A6 "EXERCISE(stage-1)" src/agentfix/agent/graph.py

### Stage 1 — see the failure first

Before editing, this fails with `NotImplementedError`. That confirms you really are on the exercise
version. Edit and save `src/agentfix/agent/graph.py`, then rerun the cell until it passes.

The tests drive the **real** graph against the **real** tools in a real temp directory — only the
model is scripted, so nothing here needs Ollama.

In [ ]:
!python -m unittest exercises.stage_1.test_stage_1 -v

### Stage 1 — optional solution peek

Reads git objects only. It does **not** check out the branch and does **not** touch your working
files.

In [ ]:
!git --no-pager diff main stage-1-solution -- src/agentfix/agent/graph.py

## Stage 2 — refusing a call the model already made

Same file, `EXERCISE(stage-2)`, inside **`tools_node`**. (Your Stage 1 routing stays — leave it
alone.)

Small models get stuck in the plainest way possible: they call `read_file` on the same path, get
the same answer, and call it again until the budget runs out. The loop guard is the policy that
breaks the pattern, and it is **yours** — LangGraph has no hook for it at all.

Already in scope: `current` (this call's signature), `signature` (the previous executed call's),
`hits` (consecutive repeats), `guard_observation(name, hits)` (the refusal text),
`tracer.note("tool", name, ...)`, and `runnable` (the calls that will actually execute).

Three things make it work:

- **What counts as the same call** — `call_signature` hashes the tool name plus its *sorted*
  arguments, so key order in the model's JSON cannot defeat the guard.
- **A refused call still gets an answer** — the API requires exactly one reply per
  `tool_call_id`. Drop one and the *next* request is rejected, one turn away from the code that
  caused it. So a refusal appends a message and moves on rather than falling through.
- **The counter moves both ways** — a repeat increments it; a call that is not a repeat resets it
  and becomes the new baseline.

The full guide is `exercises/stage_2/README.md`.

In [ ]:
!cat exercises/stage_2/README.md

In [ ]:
!echo "Stage 2 — where to edit:"
!grep -n -B12 -A8 "EXERCISE(stage-2)" src/agentfix/agent/graph.py

### Stage 2 — test

Stage 1 should already pass. Run both while you work.

In [ ]:
!python -m unittest exercises.stage_1.test_stage_1 exercises.stage_2.test_stage_2 -v

### Stage 2 — optional solution peek

This shows **only the change Stage 2 introduces**, by comparing the two solution snapshots. Your
checkout stays on `main`.

In [ ]:
!git --no-pager diff stage-1-solution stage-2-solution -- src/agentfix/agent/graph.py

### Lesson 3 — final verification

The repo's own suite, offline, with no model process anywhere.

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -5

### Run the graph for real

Two lines in the trace are yours. The run does not end on the green test result — it ends one turn
later, on the model's prose reply, because that is where Stage 1 put the `is_done` check. And if
the model gets stuck you will see `guarded — identical call #2 in a row`: the only line in a trace
that no tool produced.

In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose

In [ ]:
# All three workshop tasks. Slow — it runs the local model repeatedly.
!agentfix eval --suite workshop --limit 3

---

# Lesson 4 — What about thinking?

Same graph, driven by a model that **reasons before it acts**.

One flag on the client — `reasoning=True` — asks Ollama for the thinking and hands it back on
**its own channel** instead of leaving `<think>` tags inline in the answer. That channel matters
more than it sounds: with the tags inline, the model's deliberation ends up in the next prompt, in
the trace, and — the expensive one — inside the "complete file contents" that `write_file` is
handed.

**There is no think step and no new node.** This is what ReAct means: the model thinks and acts in
the *same* turn. The graph from lesson 3 is unchanged in shape. What changed is two decisions about
what a turn *was*:

- **A turn with no tool call is no longer rare.** The Instruct model acted on every turn but the
  last. A thinking model will spend an entire turn reasoning and ask for nothing — and nudging that
  forever is an unbounded loop wearing a step budget as a disguise. Hence `idle_turns` and
  `MAX_IDLE_TURNS = 2`: a loop guard for thinking, alongside the one for actions.
- **The action guard must ignore reasoning.** `call_signature` still hashes only the tool name and
  arguments. A model that reasons its way to the same useless call by a fresh route every time is
  still stuck, and novel thinking must not buy a repeated call another turn.

Note the CLI name changes to `agentgraph`, and the model changes to `qwen3:1.7b`.

In [ ]:
%cd /content/agentfix-react
%env MELLUM_MODEL=qwen3:1.7b

!python -m pip install -q -e .
!agentgraph doctor

Two of those checks are new, and they are the ones worth having, because both failures leave you
with a *working* agent that nothing else will complain about:

- **`reasoning`** — the model thinks, and the thinking arrives on its own channel. Fails distinctly
  if it is coming back inline as `<think>` tags.
- **`tool calling`** — it can still act while thinking. A model that reasons and calls nothing
  changes no files.

`ram` will FAIL on Colab as before; that is expected.

## Stage 1 — reasoning is not an action

Open `src/agentgraph/agent/graph.py`. Four `EXERCISE(stage-1)` markers, all one decision split four
ways:

| # | Where | What it decides |
|---|---|---|
| 1 | `acted()` | what counts as a turn that *did* something — a tool call, not prose and not thought |
| 2 | `agent_node`'s returned state | keeping `idle_turns` current |
| 3 | `nudge_node` | which of the two corrections to send |
| 4 | `route_after_agent` | the answers for a turn that acted on nothing |

Two traps worth naming before you start:

- `idle_turns` is the one key in the state with **no reducer**, because it has to *reset* — a
  reducer is handed only `(current, incoming)` and cannot tell "one more idle turn" from "that turn
  acted, start again". `agent_node` is its only writer and returns the absolute value.
- In the routing, **the verdict goes before the idle guard**. A thinking turn on a suite that is
  already green is a successful finish, not a stall. And the budget outranks the guard.

The abandonment also deserves a trace note, worded from what was observed — "no tool call", not
"turns of reasoning". A turn can ask for nothing without having reasoned, and a trace line claiming
deliberation that never happened is the exact failure this edition exists to fix.

The full guide is `exercises/stage_1/README.md`.

In [ ]:
!cat exercises/stage_1/README.md

In [ ]:
!echo "Stage 1 — all four places to edit:"
!grep -n -B6 -A4 "EXERCISE(stage-1)" src/agentgraph/agent/graph.py

### Stage 1 — test

Fails before you edit. The fake model puts its reasoning in *exactly* the field the real client
uses, so a fake that put it anywhere else would let a broken agent pass.

In [ ]:
!python -m unittest exercises.stage_1.test_stage_1 -v

In [ ]:
# The repo's own suite, offline.
!python -m unittest discover -s tests -t . 2>&1 | tail -5

### Stage 1 — optional solution peek

Git objects only; your working tree is untouched.

In [ ]:
!git --no-pager diff main stage-1-solution -- src/agentgraph/agent/graph.py

### Run the thinking agent for real

Every model turn now prints a `thinks` line above what it did. Read one — that text is the plan the
earlier editions never had. Two more things to watch for:

- `(NO REASONING)` now means what it says. In lesson 3 it appeared on almost every turn, because
  reasoning was read off `content`; here it prints only when the model genuinely skipped thinking.
- Reason twice with no tool call and the run ends with
  `abandoned — 2 consecutive turns with no tool call`. That is your Stage 1 guard.

Expect this to be **slower and more expensive per task** than lesson 3. That is the trade, and the
next cell is where you see it.

In [ ]:
!agentgraph solve tasks/workshop/01-shopcart --verbose

In [ ]:
# Slow. Reasoning is generated tokens, and prior thoughts are re-sent on every later turn.
!agentgraph eval --suite workshop --limit 3

---

# What the numbers say

Your Colab runs use 1.5B and 1.7B stand-ins, so do not read your own pass rates as the course's
result. These are the shipped measurements from the reference machine (Apple M4, 24 GB, Mellum2
12B), on 20 HumanEvalFix tasks with the same 10-step budget — each repo ships them under
`results/precomputed/`:

| Edition | pass@1 | median steps | tokens | wall clock | peak prompt |
|---|---|---|---|---|---|
| No framework, Instruct | 0.60 (12/20) | 7 | 185,235 | 8m08s | 2,998 |
| LangGraph, Instruct | 0.45 (9/20) | 10 | 237,651 | 8m15s | 3,929 |
| **LangGraph, Thinking** | **0.80 (16/20)** | **5** | **415,333** | **52m25s** | **12,599** |

**Thinking is the largest single move in the course.** It did not just solve more, it solved in
*fewer* turns — fourteen of the sixteen successes took exactly five steps. And it is expensive:
1.75× the tokens for 6× the wall clock, and a peak prompt of 12,599 against a 16,384-token
window — three-quarters of the way to overflow on a benchmark of *small* bugs.

**Do not read 0.60 → 0.45 as a cost of the framework.** Temperature is 0.6 in all three, so a
single 20-task run is noisy, and the two Instruct editions take identical step counts on the tasks
they both solve. In the no-framework edition, making the stop condition real moved pass@1 from 0.50
to 0.60 on its own — larger than the gap between those rows. What moves the number is the prompt,
the budget and the stop condition, not the plumbing.

In [ ]:
# Every repo ships its reference runs under results/precomputed/ (a live `eval` writes to
# results/, which is gitignored). Read whichever files the clone actually has.
# Braces are avoided on purpose: IPython substitutes them inside a ! command.
!for r in agentfix-workshop agentfix-langchain agentfix-react; do \
    echo "===== $r"; \
    for f in /content/$r/results/precomputed/*.json; do \
      python3 -c "import json,sys;d=json.load(open(sys.argv[1]));rs=d['results'];t=sum(x['prompt_tokens']+x['completion_tokens'] for x in rs);s=sum(x['duration_s'] for x in rs);print('  %-13s pass@1 %.2f   steps %-12s %7d tok  %2dm%02ds  peak %6d' % (d['suite'],d['pass_at_1'],[x['steps_used'] for x in rs],t,s//60,s%60,d['peak_prompt_tokens']))" "$f"; \
    done; \
  done

# Safety, briefly

The agent executes model-written code. Two boundaries, at two different layers, and neither changed
when the agent moved onto a framework or gained reasoning — confinement is a property of the tools
and the sandbox, not of the loop that calls them:

- **The tool layer confines paths.** `resolve_in_root` rejects any path that would escape the
  task's working directory *before* a read or write happens, and the write tool is constructed with
  the set of files that existed in the pristine template, so the agent cannot create a file and
  then start writing to it.
- **The sandbox confines execution.** The default backend is a hardened subprocess — stripped
  environment, resource limits, a timeout — which is **not** a security boundary. The Docker
  backend is: no network, memory/pid/CPU caps, non-root user, read-only mount.

Docker is not available in Colab, so this notebook runs the subprocess backend throughout. On your
own machine: `AGENTFIX_SANDBOX=docker` for lessons 2–3, `AGENTGRAPH_SANDBOX=docker` for lesson 4.

In [ ]:
!echo "===== path confinement ====="; sed -n '/^def resolve_in_root/,/^def /p' /content/agentfix-react/src/agentgraph/tools/fs.py | head -30

# Where to go from here

Roughly in order of what would pay off next on the numbers above:

- **Context management** — trimming or summarising old turns, or dropping stale reasoning from the
  history. The clearest gap, and what stands between this agent and a task bigger than a one-file
  bug.
- **Planning as its own phase** — the model plans inside a turn now; nothing makes it commit to a
  plan across turns or notice when it has abandoned one.
- **Reflection / self-critique** — no separate pass where the model reviews its own diff before the
  tests do.
- **Parallel tool calls** — one at a time here, on purpose: `max_concurrency=1` is what keeps the
  test result honest. Doing it properly means knowing which calls are safe to overlap.
- **Multi-agent coordination** — one model, one graph, no delegation.

If you take one thing from the whole course, make it the stop condition: three editions in, the
thing that decides whether an agent is trustworthy is still that it believes the test suite rather
than the model.

The three repositories, if you would rather read them in your own IDE:

- https://github.com/jelenadjuric01/agentfix-workshop
- https://github.com/jelenadjuric01/agentfix-langchain
- https://github.com/jelenadjuric01/agentfix-react